In [1]:
catalogo = [
    ("Toyota Hilux 2024", "camioneta pickup con motor turbodiesel, traccion 4x4, ideal para trabajo pesado y offroad"),
    ("Ford Ranger 2024", "pickup mediana con motor turbo y chasis reforzado, disenada para terrenos dificiles"),
    ("Nissan Leaf 2024", "hatchback 100% electrico, silencioso y eficiente, pensado para uso urbano diario"),
    ("Chevrolet Bolt EV 2024", "auto electrico compacto, carga rapida, bajo costo de mantenimiento, ideal para la ciudad"),
    ("BMW X5 2024", "suv premium con interior de cuero, tecnologia avanzada y motor de alto rendimiento"),
    ("Mercedes-Benz GLE 2024", "suv de lujo con suspension adaptativa y acabados premium"),
    ("Jeep Wrangler 2024", "todoterreno 4x4 con carroceria removible, disenado para exploracion extrema en montana y rios"),
    ("Land Rover Defender 2024", "todoterreno robusto con vadeo profundo, construido para climas humedos y terrenos extremos"),
]

In [2]:
pares = []
for i, (title_i, body_i) in enumerate(catalogo):
    pares.append((title_i, body_i, 1))
    for j, (title_j, body_j) in enumerate(catalogo):
        if i != j:
            pares.append((title_i, body_j, 0))

print(f"Total de pares: {len(pares)}")
print(pares[0])

Total de pares: 64
('Toyota Hilux 2024', 'camioneta pickup con motor turbodiesel, traccion 4x4, ideal para trabajo pesado y offroad', 1)


In [3]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

textos_entrenamiento = [texto_a + " " + texto_b for texto_a, texto_b, _ in pares]

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    special_tokens=["[UNK]"],
    vocab_size=500,
    min_frequency=2,
)

tokenizer.train_from_iterator(textos_entrenamiento, trainer)

print("Vocabulario final:", tokenizer.get_vocab_size())

Vocabulario final: 357


In [4]:
for palabra in ["camioneta", "motor", "electrico", "premium", "wayrapiquero", "wayrasacha"]:
    output = tokenizer.encode(palabra)
    print(f"{palabra:14} -> {output.tokens}")

camioneta      -> ['camioneta']
motor          -> ['motor']
electrico      -> ['electrico']
premium        -> ['premium']
wayrapiquero   -> ['[UNK]', 'a', 'y', 'ra', 'p', 'i', '[UNK]', 'u', 'ero']
wayrasacha     -> ['[UNK]', 'a', 'y', 'ra', 's', 'a', 'ch', 'a']


In [ ]:
import torch
import torch.nn as nn

# ---- Definir las piezas (pesos), sueltas, no dentro de una clase ----
d, d_ff = 16, 32

Wq = nn.Linear(d, d, bias=False)
Wk = nn.Linear(d, d, bias=False)
Wv = nn.Linear(d, d, bias=False)

In [ ]:
print(Wq.weight.shape) 

In [ ]:
norm1 = nn.LayerNorm(d)
norm1


In [ ]:
print(norm1.weight.shape) 

In [ ]:
print(norm1.weight.shape)

In [ ]:
print(norm1.bias.shape) 

In [ ]:
ff = nn.Sequential(nn.Linear(d, d_ff), nn.ReLU(), nn.Linear(d_ff, d))

In [ ]:
ff

In [ ]:
norm2 = nn.LayerNorm(d)
print(norm1.weight.shape)

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

lin1 = nn.Linear(2, 3, bias=False)
relu = nn.ReLU()
lin2 = nn.Linear(3, 2, bias=False)

lin1.weight.data = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])
lin2.weight.data = torch.tensor([
    [1.0, 0.0, 0.3],
    [0.0, 1.0, 0.3],
])

xs = torch.linspace(-2, 2, 13)
ys = torch.linspace(-2, 2, 13)
xx, yy = torch.meshgrid(xs, ys, indexing="ij")
pts = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
color = pts[:, 0]

h = lin1(pts).detach()
h_relu = relu(h).detach()
out = lin2(h_relu).detach()

fig = plt.figure(figsize=(14, 4))

ax = fig.add_subplot(1, 4, 1)
ax.scatter(pts[:, 0], pts[:, 1], c=color, cmap="coolwarm", s=18)
ax.set_title("input (2D)")
ax.set_aspect("equal")
ax.axhline(0, color="k", lw=0.4)
ax.axvline(0, color="k", lw=0.4)

ax = fig.add_subplot(1, 4, 2, projection="3d")
ax.scatter(h[:, 0], h[:, 1], h[:, 2], c=color, cmap="coolwarm", s=18)
ax.set_title("after Linear 2->3")

ax = fig.add_subplot(1, 4, 3, projection="3d")
ax.scatter(h_relu[:, 0], h_relu[:, 1], h_relu[:, 2], c=color, cmap="coolwarm", s=18)
ax.set_title("after ReLU")

ax = fig.add_subplot(1, 4, 4)
ax.scatter(out[:, 0], out[:, 1], c=color, cmap="coolwarm", s=18)
ax.set_title("after Linear 3->2")
ax.set_aspect("equal")
ax.axhline(0, color="k", lw=0.4)
ax.axvline(0, color="k", lw=0.4)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

palabras = ["premium", "caro", "barato"]
E = torch.tensor([
    [ 1.0,  0.8],   # premium
    [ 0.9,  0.7],   # caro
    [-1.0, -0.6],   # barato
])

lin1 = nn.Linear(2, 3, bias=False)
relu = nn.ReLU()
lin2 = nn.Linear(3, 2, bias=False)

# (x, y) -> (x, y, x+y)
lin1.weight.data = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])
# (x, y, z) -> (x + 0.3z, y + 0.3z)
lin2.weight.data = torch.tensor([
    [1.0, 0.0, 0.3],
    [0.0, 1.0, 0.3],
])

h = lin1(E).detach()
h_relu = relu(h).detach()
out = lin2(h_relu).detach()

for i, w in enumerate(palabras):
    print(f"{w:8}  in={E[i].tolist()}  lin1={h[i].tolist()}  relu={h_relu[i].tolist()}  lin2={out[i].tolist()}")

fig = plt.figure(figsize=(16, 4))

ax = fig.add_subplot(1, 4, 1)
ax.scatter(E[:, 0], E[:, 1], s=80)
for w, v in zip(palabras, E):
    ax.annotate(w, (float(v[0]), float(v[1])))
    ax.arrow(0, 0, float(v[0]), float(v[1]), head_width=0.05, length_includes_head=True)
ax.set_title("1. embeddings (2D)")
ax.set_aspect("equal")
ax.axhline(0, color="k", lw=0.4)
ax.axvline(0, color="k", lw=0.4)

ax = fig.add_subplot(1, 4, 2, projection="3d")
ax.scatter(h[:, 0], h[:, 1], h[:, 2], s=80)
for w, v in zip(palabras, h):
    ax.text(float(v[0]), float(v[1]), float(v[2]), w)
ax.set_title("2. Linear 2->3")

ax = fig.add_subplot(1, 4, 3, projection="3d")
ax.scatter(h_relu[:, 0], h_relu[:, 1], h_relu[:, 2], s=80)
for w, v in zip(palabras, h_relu):
    ax.text(float(v[0]), float(v[1]), float(v[2]), w)
ax.set_title("3. ReLU")

ax = fig.add_subplot(1, 4, 4)
ax.scatter(out[:, 0], out[:, 1], s=80)
for w, v in zip(palabras, out):
    ax.annotate(w, (float(v[0]), float(v[1])))
    ax.arrow(0, 0, float(v[0]), float(v[1]), head_width=0.05, length_includes_head=True)
ax.set_title("4. Linear 3->2")
ax.set_aspect("equal")
ax.axhline(0, color="k", lw=0.4)
ax.axvline(0, color="k", lw=0.4)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

palabras = ["premium", "caro", "barato"]
E = torch.tensor([
    [0.60, 0.50],  # premium
    [0.55, 0.55],  # caro     almost the same point
    [0.50, 0.40],  # barato   also in the same blob
])

lin1 = nn.Linear(2, 2)
relu = nn.ReLU()
lin2 = nn.Linear(2, 2, bias=False)

# Linear 1: keep x, and score = 2x + 2y - 2  (a "luxury" axis)
lin1.weight.data = torch.tensor([
    [1.0, 0.0],
    [2.0, 2.0],
])
lin1.bias.data = torch.tensor([0.0, -2.0])

# Linear 2: stretch that score x5
lin2.weight.data = torch.tensor([
    [1.0, 0.0],
    [0.0, 5.0],
])

h = lin1(E).detach()
h_relu = relu(h).detach()
out = lin2(h_relu).detach()

def cos(a, b):
    return float(torch.dot(a, b) / (a.norm() * b.norm()))

print("cosine BEFORE  premium-caro", cos(E[0], E[1]), "  premium-barato", cos(E[0], E[2]))
print("cosine AFTER   premium-caro", cos(out[0], out[1]), "  premium-barato", cos(out[0], out[2]))
for i, w in enumerate(palabras):
    print(f"{w:8}  in={E[i].tolist()}  lin1={h[i].tolist()}  relu={h_relu[i].tolist()}  out={out[i].tolist()}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
stages = [
    (E, "1. start (clustered)"),
    (h, "2. Linear (luxury axis)"),
    (h_relu, "3. ReLU (clip negatives)"),
    (out, "4. Linear (stretch)"),
]
for ax, (pts, title) in zip(axes, stages):
    ax.scatter(pts[:, 0], pts[:, 1], s=80)
    for w, v in zip(palabras, pts):
        ax.annotate(w, (float(v[0]), float(v[1])))
    ax.axhline(0, color="k", lw=0.4)
    ax.axvline(0, color="k", lw=0.4)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xlim(-0.2, 1.2)
    ax.set_ylim(-0.5, 1.2)
plt.tight_layout()
plt.show()

In [ ]:
vocab_size = tokenizer.get_vocab_size()
max_len = 30

token_embed = nn.Embedding(vocab_size, d)
pos_embed = nn.Embedding(max_len, d)


In [ ]:
tokenizer.get_vocab()

In [ ]:
vocab_size

In [ ]:
token_embed

In [ ]:
pos_embed

In [ ]:
texto = catalogo[0][1]  # "camioneta pickup con motor turbodiesel..."
ids = torch.tensor(tokenizer.encode(texto).ids)
positions = torch.arange(len(ids))

print("ids:", ids)
print("positions:", positions)

In [ ]:
texto = catalogo[0][1]
ids = torch.tensor(tokenizer.encode(texto).ids)
positions = torch.arange(len(ids))

In [ ]:
x = token_embed(ids) + pos_embed(positions)      # 1. entrada: token + posicion
x

In [ ]:
Q, K, V = Wq(x), Wk(x), Wv(x)                     # 2-4. proyecciones
Q

In [ ]:
import matplotlib.pyplot as plt

tokens = tokenizer.encode(texto).tokens

fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=True)
for ax, mat, title in zip(axes, [x, Q, K, V], ["x", "Q = Wq(x)", "K = Wk(x)", "V = Wv(x)"]):
    im = ax.imshow(mat.detach().numpy(), cmap="RdBu", vmin=-2, vmax=2, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("dim 0-15")
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=8)
axes[0].set_ylabel("token")
fig.colorbar(im, ax=axes, fraction=0.02)
plt.suptitle("same tokens, 3 different mixes of the 16 dims")
plt.show()

In [ ]:
scores = Q @ K.transpose(-2, -1) / (d ** 0.5)     # 5. scores

In [ ]:
import matplotlib.pyplot as plt

tokens = tokenizer.encode(texto).tokens

plt.figure(figsize=(7, 6))
plt.imshow(scores.detach().numpy(), cmap="RdBu", vmin=-2, vmax=2)
plt.colorbar(label="score")
plt.xticks(range(len(tokens)), tokens, rotation=90, fontsize=8)
plt.yticks(range(len(tokens)), tokens, fontsize=8)
plt.xlabel("key (what I look at)")
plt.ylabel("query (who is looking)")
plt.title("scores = Q @ K.T / sqrt(d)")
plt.tight_layout()
plt.show()

In [ ]:
weights = torch.softmax(scores, dim=-1)           # 6. softmax

In [ ]:
import matplotlib.pyplot as plt

tokens = tokenizer.encode(texto).tokens

plt.figure(figsize=(7, 6))
plt.imshow(weights.detach().numpy(), cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="weight")
plt.xticks(range(len(tokens)), tokens, rotation=90, fontsize=8)
plt.yticks(range(len(tokens)), tokens, fontsize=8)
plt.xlabel("key (who I attend to)")
plt.ylabel("query (who is attending)")
plt.title("weights = softmax(scores)")
plt.tight_layout()
plt.show()

print(weights.sum(dim=-1).detach().numpy())  # ~1.0 per row

In [ ]:
attn_out = weights @ V                            # 7. mezcla ponderada

In [ ]:
import matplotlib.pyplot as plt

tokens = tokenizer.encode(texto).tokens

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
for ax, mat, title in zip(axes, [V, attn_out], ["V (before mix)", "attn_out = weights @ V"]):
    im = ax.imshow(mat.detach().numpy(), cmap="RdBu", vmin=-2, vmax=2, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("dim 0-15")
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=8)
axes[0].set_ylabel("token")
fig.colorbar(im, ax=axes, fraction=0.03)
plt.show()

In [ ]:
i = 0  # first token
print(tokens[i], "looks at:")
for w, t in zip(weights[i].detach(), tokens):
    print(f"  {t:16} {float(w):.3f}")

print("V mix:", (weights[i] @ V).detach().numpy())
print("attn_out[i]:", attn_out[i].detach().numpy())  # same numbers

In [ ]:
x = norm1(x + attn_out)      

In [ ]:
import matplotlib.pyplot as plt

tokens = tokenizer.encode(texto).tokens

x_in = x                         # keep the original
residual = x_in + attn_out       # skip connection
x_out = norm1(residual)          # then LayerNorm
# later you can do: x = x_out

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for ax, mat, title in zip(
    axes,
    [x_in, residual, x_out],
    ["x (before)", "x + attn_out", "norm1(x + attn_out)"],
):
    im = ax.imshow(mat.detach().numpy(), cmap="RdBu", vmin=-2, vmax=2, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("dim 0-15")
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=8)
axes[0].set_ylabel("token")
fig.colorbar(im, ax=axes, fraction=0.02)
plt.show()

print("mean per token, before:", x_in.mean(dim=-1).detach().numpy())
print("mean per token, after: ", x_out.mean(dim=-1).detach().numpy())  # ~0

In [ ]:
ff_out = ff(x)                                    # 9. feed-forward

In [ ]:
import matplotlib.pyplot as plt

tokens = tokenizer.encode(texto).tokens

h = ff[0](x).detach()          # Linear 16 → 32
h_relu = ff[1](h).detach()     # ReLU
ff_out = ff[2](h_relu).detach()  # Linear 32 → 16

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for ax, mat, title in zip(
    axes,
    [x, h_relu, ff_out],
    ["x in (16)", "after Linear+ReLU (32)", "ff_out (16)"],
):
    im = ax.imshow(mat.detach().numpy(), cmap="RdBu", vmin=-2, vmax=2, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("dim")
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=8)
axes[0].set_ylabel("token")
fig.colorbar(im, ax=axes, fraction=0.02)
plt.show()

print("zeros after ReLU:", int((h_relu == 0).sum()), "/", h_relu.numel())

In [ ]:
x = norm2(x + ff_out)    

In [ ]:
import matplotlib.pyplot as plt

tokens = tokenizer.encode(texto).tokens

x_ff_in = x                      # after attention + norm1
residual2 = x_ff_in + ff_out
x_ff_out = norm2(residual2)
# later: x = x_ff_out

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for ax, mat, title in zip(
    axes,
    [x_ff_in, residual2, x_ff_out],
    ["x (after attn)", "x + ff_out", "norm2(x + ff_out)"],
):
    im = ax.imshow(mat.detach().numpy(), cmap="RdBu", vmin=-2, vmax=2, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("dim 0-15")
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=8)
axes[0].set_ylabel("token")
fig.colorbar(im, ax=axes, fraction=0.02)
plt.show()

print("mean per token, before:", x_ff_in.mean(dim=-1).detach().numpy())
print("mean per token, after: ", x_ff_out.mean(dim=-1).detach().numpy())  # ~0

In [ ]:
import torch
import torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.Wq = nn.Linear(d, d, bias=False)
        self.Wk = nn.Linear(d, d, bias=False)
        self.Wv = nn.Linear(d, d, bias=False)
        self.norm1 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, d_ff), nn.ReLU(), nn.Linear(d_ff, d))
        self.norm2 = nn.LayerNorm(d)

    def forward(self, x):
        Q, K, V = self.Wq(x), self.Wk(x), self.Wv(x)
        scores = Q @ K.transpose(-2, -1) / (x.shape[-1] ** 0.5)
        weights = torch.softmax(scores, dim=-1)
        x = self.norm1(x + weights @ V)
        x = self.norm2(x + self.ff(x))
        return x


class EmbeddingModel(nn.Module):
    def __init__(self, vocab_size, d, d_ff, max_len, n_blocks):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d)
        self.pos_embed = nn.Embedding(max_len, d)
        self.blocks = nn.ModuleList([TransformerBlock(d, d_ff) for _ in range(n_blocks)])

    def forward(self, token_ids):
        positions = torch.arange(len(token_ids))
        x = self.token_embed(token_ids) + self.pos_embed(positions)
        for block in self.blocks:
            x = block(x)
        return x.mean(dim=0)

In [ ]:
vocab_size = tokenizer.get_vocab_size()
model = EmbeddingModel(vocab_size=vocab_size, d=16, d_ff=32, max_len=30, n_blocks=2)

In [ ]:
# ---- 2. Tokenizador BPE entrenado sobre el corpus ----
textos_entrenamiento = [texto_a + " " + texto_b for texto_a, texto_b, _ in pares]

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(special_tokens=["[UNK]"], vocab_size=500, min_frequency=2)
tokenizer.train_from_iterator(textos_entrenamiento, trainer)

# ---- 3. Modelo ----
class TransformerBlock(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.Wq = nn.Linear(d, d, bias=False)
        self.Wk = nn.Linear(d, d, bias=False)
        self.Wv = nn.Linear(d, d, bias=False)
        self.norm1 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, d_ff), nn.ReLU(), nn.Linear(d_ff, d))
        self.norm2 = nn.LayerNorm(d)

    def forward(self, x):
        Q, K, V = self.Wq(x), self.Wk(x), self.Wv(x)
        scores = Q @ K.transpose(-2, -1) / (x.shape[-1] ** 0.5)
        weights = torch.softmax(scores, dim=-1)
        x = self.norm1(x + weights @ V)
        x = self.norm2(x + self.ff(x))
        return x

class EmbeddingModel(nn.Module):
    def __init__(self, vocab_size, d, d_ff, max_len, n_blocks):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d)
        self.pos_embed = nn.Embedding(max_len, d)
        self.blocks = nn.ModuleList([TransformerBlock(d, d_ff) for _ in range(n_blocks)])

    def forward(self, token_ids):
        positions = torch.arange(len(token_ids))
        x = self.token_embed(token_ids) + self.pos_embed(positions)
        for block in self.blocks:
            x = block(x)
        return x.mean(dim=0)

def texto_a_ids(texto, tokenizer, max_len):
    ids = tokenizer.encode(texto).ids[:max_len]   # <-- CAMBIO: trunca por seguridad
    return torch.tensor(ids)

def contrastive_loss(vec_a, vec_b, label, margin=0.3):
    cos_sim = nn.functional.cosine_similarity(vec_a.unsqueeze(0), vec_b.unsqueeze(0))
    if label == 1:
        return 1 - cos_sim
    else:
        return torch.clamp(cos_sim - margin, min=0)

# ---- 4. Instanciar y entrenar ----
MAX_LEN = 64   # <-- CAMBIO: antes 30, subido para que quepan textos con Wayra fragmentado

vocab_size = tokenizer.get_vocab_size()
model = EmbeddingModel(vocab_size=vocab_size, d=16, d_ff=32, max_len=MAX_LEN, n_blocks=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(200):
    total_loss = 0.0
    for texto_a, texto_b, label in pares:
        ids_a = texto_a_ids(texto_a, tokenizer, MAX_LEN)
        ids_b = texto_a_ids(texto_b, tokenizer, MAX_LEN)
        vec_a = model(ids_a)
        vec_b = model(ids_b)
        loss = contrastive_loss(vec_a, vec_b, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 20 == 0:
        print(f"epoch {epoch:4d}  loss {total_loss/len(pares):.4f}")


In [ ]:
wayrapiquero = "wayrapiquero auto electrico premium para la isla, silencioso y elegante"
wayrasacha = "wayrasacha camioneta a gasolina, resistente y aventurera para la selva"

with torch.no_grad():
    vec_piquero = model(texto_a_ids(wayrapiquero, tokenizer, MAX_LEN))
    vec_sacha = model(texto_a_ids(wayrasacha, tokenizer, MAX_LEN))
    vecs_catalogo = {title: model(texto_a_ids(body, tokenizer, MAX_LEN)) for title, body in catalogo}

def cos(a, b):
    return nn.functional.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

print("\n=== Wayra Piquero vs catalogo ===")
for title, vec in vecs_catalogo.items():
    print(f"  {title:28} {cos(vec_piquero, vec):.3f}")

print("\n=== Wayra Sacha vs catalogo ===")
for title, vec in vecs_catalogo.items():
    print(f"  {title:28} {cos(vec_sacha, vec):.3f}")

In [ ]:
# ---- 1. Documentos a indexar: catalogo real + los productos Wayra ----
documentos = {title: body for title, body in catalogo}
documentos["Wayra Piquero"] = "wayrapiquero auto electrico premium para la isla, silencioso y elegante"
documentos["Wayra Sacha"] = "wayrasacha camioneta a gasolina, resistente y aventurera para la selva"

# ---- 2. Almacenar: generar y guardar UN embedding por documento ----
with torch.no_grad():
    index = {nombre: model(texto_a_ids(texto, tokenizer, MAX_LEN)) for nombre, texto in documentos.items()}


In [ ]:
index

In [ ]:
def cos(a, b):
    return nn.functional.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

def buscar(query, index, model, tokenizer, max_len, topk=3):
    with torch.no_grad():
        vec_query = model(texto_a_ids(query, tokenizer, max_len))
    resultados = [(nombre, cos(vec_query, vec)) for nombre, vec in index.items()]
    return sorted(resultados, key=lambda x: -x[1])[:topk]

In [ ]:
resultados = buscar("auto electrico", index, model, tokenizer, MAX_LEN)
for nombre, score in resultados:
    print(f"{nombre:28} {score:.3f}")

In [ ]:
documentos

In [ ]:
with torch.no_grad():
    vec_leaf = model(texto_a_ids(documentos["Nissan Leaf 2024"], tokenizer, MAX_LEN))
    vec_ranger = model(texto_a_ids(documentos["Ford Ranger 2024"], tokenizer, MAX_LEN))
    vec_query = model(texto_a_ids("auto electrico", tokenizer, MAX_LEN))

print("query vs Leaf:  ", cos(vec_query, vec_leaf))
print("query vs Ranger:", cos(vec_query, vec_ranger))